# 06 — Post-Normalisation Audit

**Goal.** A quality gate before modelling. Are the Full and Reduced branches complete, valid, and **identical apart from the face**? It is descriptive only: nothing is fitted or transformed.

**Approach**
- Independent disk inventory and coherent discovery of both branch outputs. Clip, label, signer and frame-count agreement for all 6,000 clips.
- Schemas checked against an **independent hard-coded spec** (not trusted from the files). Widths, finiteness and binary masks are checked for every array.
- **Shared-tail check:** `full[:, face_end:]` vs `reduced[:, face_end:]` compared numerically *and* byte-level for every clip. This turns "only the face differs" from an argument into a measurement.
- Synthetic corruption tests confirm the checks actually fail on bad data. Smoke and full modes are available. Statistics are published only after a full-mode PASS.

**Key design decision: per-signer sufficient statistics.** Because this notebook already reads every file, it records each signer's frame count, column sums and sums of squares (float64). A fold's standardiser is then just the sum over its 12 training signers. The arithmetic is identical, it avoids 16 extra passes over ~7 GB, and a fold structurally **cannot** touch test-signer data. Presence masks are excluded from standardisation.

**Result.** PASS: 6,000 clips × 806,159 frames in both branches, widths 1,759 / 631, identical 355-column tail.

**Inputs → Outputs.** 05a + 05b outputs → `post_normalization_audit.{json,csv}`, `signer_feature_stats_{full,reduced}.npz`, `signer_quality_summary.csv`, `branch_alignment_report.csv`, diagnostic plots.

In [ ]:
# ============================================================
# CELL 1 — Configuration, environment, expected layout
# ============================================================
from __future__ import annotations

import hashlib
import json
import multiprocessing
import os
import platform
import random
import shutil
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

AUDIT_VERSION = "nb6-v3"

# ---- Run mode -------------------------------------------------------------
SMOKE_TEST = False
SMOKE_TEST_CLIPS = 200
OVERWRITE = False
CHECKPOINT_EVERY = 500

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---- Manual overrides: all three per branch, or none ----------------------
OVERRIDE_FULL_DIR: Optional[str] = None
OVERRIDE_FULL_METADATA: Optional[str] = None
OVERRIDE_FULL_SCHEMA: Optional[str] = None
OVERRIDE_REDUCED_DIR: Optional[str] = None
OVERRIDE_REDUCED_METADATA: Optional[str] = None
OVERRIDE_REDUCED_SCHEMA: Optional[str] = None

# ---- Expected invariants --------------------------------------------------
EXPECTED_CLIPS = 6000
EXPECTED_LABELS = 200
EXPECTED_SIGNERS = 16
LONG_CLIP_THRESHOLD = 256
EXPECTED_PIPELINE_ID = "24c9d5ccb3f1"

# The layout this experiment is DEFINED to have, stated independently of the
# attached schema files. Validating a schema only against itself is circular:
# a wrong schema paired with equally wrong arrays would agree and pass.
EXPECTED_LAYOUT: Dict[str, Dict[str, Any]] = {
    "full": {
        "features_per_frame": 1759,
        "face_landmarks_kept": 468,
        "blocks": [
            ("face", 0, 1404), ("pose", 1404, 1503),
            ("left_hand", 1503, 1566), ("right_hand", 1566, 1629),
            ("left_hand_local", 1629, 1692), ("right_hand_local", 1692, 1755),
            ("presence_mask", 1755, 1759),
        ],
    },
    "reduced": {
        "features_per_frame": 631,
        "face_landmarks_kept": 92,
        "blocks": [
            ("face", 0, 276), ("pose", 276, 375),
            ("left_hand", 375, 438), ("right_hand", 438, 501),
            ("left_hand_local", 501, 564), ("right_hand_local", 564, 627),
            ("presence_mask", 627, 631),
        ],
    },
}

# ---- Files ----------------------------------------------------------------
WORKING = Path("/kaggle/working")

# Raw resume checkpoint: ONLY the direct audit records. Kept separate from the
# final table so a resume never re-merges metadata columns onto themselves.
RAW_CHECKPOINT = WORKING / "_post_normalization_checkpoint.csv"
ACCUMULATOR_CHECKPOINT = WORKING / "_signer_statistics_checkpoint.npz"

AUDIT_CSV = WORKING / "post_normalization_audit.csv"          # final, enriched
AUDIT_JSON = WORKING / "post_normalization_audit.json"
SIGNER_CSV = WORKING / "signer_quality_summary.csv"
ALIGNMENT_CSV = WORKING / "branch_alignment_report.csv"
PLOT_DIR = WORKING / "audit_plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Statistics are written unverified first and published only after a PASS.
UNVERIFIED_STATS = {branch: WORKING / f"_unverified_signer_feature_stats_{branch}.npz"
                    for branch in ("full", "reduced")}
PUBLISHED_STATS = {branch: WORKING / f"signer_feature_stats_{branch}.npz"
                   for branch in ("full", "reduced")}

NUM_WORKERS = min(4, os.cpu_count() or 1)

ENVIRONMENT = {
    "audit_version": AUDIT_VERSION, "python": platform.python_version(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__, "cpu_count": os.cpu_count(),
    "workers": NUM_WORKERS, "seed": RANDOM_SEED, "smoke_test": SMOKE_TEST,
}


def atomic_write(path: Path, writer: Callable[[Any], None]) -> None:
    """Write to a temporary file then rename.

    The writer receives an open binary handle, not a path: np.save and
    np.savez_compressed silently append their own extension to a filename
    lacking it, which would rename a file that was never created.
    """
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")
print(f"\nMode: {'SMOKE TEST (' + str(SMOKE_TEST_CLIPS) + ' clips)' if SMOKE_TEST else 'FULL RUN'}")
print(f"Overwrite: {OVERWRITE}")

In [ ]:
# ============================================================
# CELL 2 — Bundle discovery, override coherence, schema validation
# ============================================================

def bundle_paths(root: Path, branch: str) -> Dict[str, Path]:
    """The three artefacts a Phase 5 run writes, relative to one bundle root."""
    return {
        "root": root,
        "landmarks": root / "normalized_landmarks" / branch,
        "metadata": root / f"normalized_metadata_{branch}.csv",
        "schema": root / f"normalized_feature_schema_{branch}.json",
    }


def find_bundles(branch: str) -> List[Dict[str, Path]]:
    root = Path("/kaggle/input")
    if not root.exists():
        raise FileNotFoundError("/kaggle/input does not exist. Attach the Phase 5 outputs.")
    found, seen = [], set()
    for landmark_dir in sorted(root.glob(f"**/normalized_landmarks/{branch}")):
        if not landmark_dir.is_dir():
            continue
        candidate = bundle_paths(landmark_dir.parent.parent, branch)
        if (candidate["metadata"].is_file() and candidate["schema"].is_file()
                and candidate["root"] not in seen):
            seen.add(candidate["root"])
            found.append(candidate)
    return found


def check_override_coherence(branch: str, landmarks: Path, metadata: Path,
                             schema: Path) -> List[str]:
    """Manual paths must still form one structurally valid bundle.

    Supplying three existing files is not enough: they could come from three
    different attached inputs, which is exactly the mismatch auto-discovery
    was hardened against.
    """
    problems: List[str] = []
    landmarks = landmarks.resolve()
    if landmarks.name != branch or landmarks.parent.name != "normalized_landmarks":
        problems.append(
            f"landmark dir must end with normalized_landmarks/{branch}, got {landmarks}")
        return problems

    expected = bundle_paths(landmarks.parent.parent, branch)
    for key, given in (("metadata", metadata), ("schema", schema)):
        if given.resolve() != expected[key].resolve():
            problems.append(
                f"{key} is not in the same bundle: expected {expected[key]}, got {given.resolve()}")
    return problems


def resolve_branch(branch: str, override_dir: Optional[str],
                   override_metadata: Optional[str],
                   override_schema: Optional[str]) -> Dict[str, Path]:
    provided = [override_dir, override_metadata, override_schema]
    if any(provided):
        if not all(provided):
            raise ValueError(
                f"For '{branch}', override all three paths or none. Overriding some "
                "reintroduces the mismatched-bundle risk this guards against.")
        landmarks, metadata, schema = (Path(override_dir), Path(override_metadata),
                                       Path(override_schema))
        for label, path in (("landmark dir", landmarks), ("metadata", metadata),
                            ("schema", schema)):
            if not path.exists():
                raise FileNotFoundError(f"{branch} override {label} not found: {path}")
        problems = check_override_coherence(branch, landmarks, metadata, schema)
        if problems:
            raise RuntimeError(
                f"Incoherent manual override for '{branch}':\n  " + "\n  ".join(problems))
        print(f"  {branch}: manual overrides accepted (bundle coherent)")
        return bundle_paths(landmarks.resolve().parent.parent, branch)

    found = find_bundles(branch)
    if not found:
        raise FileNotFoundError(
            f"No coherent '{branch}' bundle under /kaggle/input. A bundle needs:\n"
            f"  <root>/normalized_landmarks/{branch}/\n"
            f"  <root>/normalized_metadata_{branch}.csv\n"
            f"  <root>/normalized_feature_schema_{branch}.json")
    if len(found) > 1:
        listing = "\n".join(f"    {b['root']}" for b in found)
        raise RuntimeError(
            f"{len(found)} '{branch}' bundles attached; refusing to guess.\n{listing}\n"
            "Detach the extras, or set the OVERRIDE_ variables.")
    print(f"  {branch}: auto-discovered at {found[0]['root']}")
    return found[0]


def validate_schema(branch: str, schema: Dict[str, Any]) -> List[str]:
    """Check a schema against EXPECTED_LAYOUT, not against itself."""
    expected = EXPECTED_LAYOUT[branch]
    problems: List[str] = []

    width = schema.get("features_per_frame")
    if width != expected["features_per_frame"]:
        problems.append(f"features_per_frame {width} != {expected['features_per_frame']}")
    if schema.get("face_landmarks_kept") != expected["face_landmarks_kept"]:
        problems.append(
            f"face_landmarks_kept {schema.get('face_landmarks_kept')} "
            f"!= {expected['face_landmarks_kept']}")
    if schema.get("pipeline_id") != EXPECTED_PIPELINE_ID:
        problems.append(f"pipeline_id {schema.get('pipeline_id')} != {EXPECTED_PIPELINE_ID}")

    blocks = schema.get("blocks", [])
    if len(blocks) != len(expected["blocks"]):
        problems.append(f"{len(blocks)} blocks, expected {len(expected['blocks'])}")
        return problems

    cursor = 0
    for index, (name, start, end) in enumerate(expected["blocks"]):
        actual = blocks[index]
        if actual.get("block") != name:
            problems.append(f"block {index}: name '{actual.get('block')}' != '{name}'")
        if int(actual.get("start", -1)) != start:
            problems.append(f"block '{name}': start {actual.get('start')} != {start}")
        if int(actual.get("end_exclusive", -1)) != end:
            problems.append(f"block '{name}': end {actual.get('end_exclusive')} != {end}")
        if int(actual.get("start", -1)) != cursor:
            problems.append(f"block '{name}': gap or overlap at {actual.get('start')} "
                            f"(previous block ended at {cursor})")
        cursor = int(actual.get("end_exclusive", cursor))

    if cursor != expected["features_per_frame"]:
        problems.append(f"final block ends at {cursor}, expected {expected['features_per_frame']}")

    mask = next((b for b in blocks if b.get("block") == "presence_mask"), None)
    if mask is None:
        problems.append("presence_mask block missing")
    elif int(mask["end_exclusive"]) - int(mask["start"]) != 4:
        problems.append(
            f"presence_mask spans {int(mask['end_exclusive']) - int(mask['start'])} columns, expected 4")
    return problems


print("Resolving branch bundles")
FULL_BUNDLE = resolve_branch("full", OVERRIDE_FULL_DIR, OVERRIDE_FULL_METADATA,
                             OVERRIDE_FULL_SCHEMA)
REDUCED_BUNDLE = resolve_branch("reduced", OVERRIDE_REDUCED_DIR, OVERRIDE_REDUCED_METADATA,
                                OVERRIDE_REDUCED_SCHEMA)

FULL_DIR, REDUCED_DIR = FULL_BUNDLE["landmarks"], REDUCED_BUNDLE["landmarks"]
FULL_METADATA_PATH, REDUCED_METADATA_PATH = FULL_BUNDLE["metadata"], REDUCED_BUNDLE["metadata"]
FULL_SCHEMA_PATH, REDUCED_SCHEMA_PATH = FULL_BUNDLE["schema"], REDUCED_BUNDLE["schema"]

print("\nResolved paths")
for label, bundle in (("full", FULL_BUNDLE), ("reduced", REDUCED_BUNDLE)):
    for key in ("root", "landmarks", "metadata", "schema"):
        print(f"  [{label}] {key:<10} {bundle[key]}")


def file_hash(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()[:16]


METADATA_HASHES = {"full": file_hash(FULL_METADATA_PATH),
                   "reduced": file_hash(REDUCED_METADATA_PATH)}
SCHEMA_HASHES = {"full": file_hash(FULL_SCHEMA_PATH),
                 "reduced": file_hash(REDUCED_SCHEMA_PATH)}

FULL_SCHEMA = json.loads(FULL_SCHEMA_PATH.read_text(encoding="utf-8"))
REDUCED_SCHEMA = json.loads(REDUCED_SCHEMA_PATH.read_text(encoding="utf-8"))

schema_problems = {"full": validate_schema("full", FULL_SCHEMA),
                   "reduced": validate_schema("reduced", REDUCED_SCHEMA)}
print("\nSchema validation against the independent expected layout")
for branch, problems in schema_problems.items():
    if problems:
        print(f"  {branch}: {len(problems)} PROBLEM(S)")
        for problem in problems:
            print(f"    - {problem}")
    else:
        print(f"  {branch}: OK ({EXPECTED_LAYOUT[branch]['features_per_frame']} features, "
              f"{len(EXPECTED_LAYOUT[branch]['blocks'])} blocks)")

SCHEMAS_VALID = not any(schema_problems.values())
if not SCHEMAS_VALID:
    raise RuntimeError(
        "Schema validation failed. The attached data does not match the layout this "
        "experiment is defined to have; nothing below would be meaningful.")


def block_layout_hash(schema: Dict[str, Any]) -> str:
    payload = json.dumps(
        [[b["block"], int(b["start"]), int(b["end_exclusive"])] for b in schema["blocks"]],
        separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


BLOCK_LAYOUT_HASHES = {"full": block_layout_hash(FULL_SCHEMA),
                       "reduced": block_layout_hash(REDUCED_SCHEMA)}


def block_bounds(schema: Dict[str, Any], block_name: str) -> Tuple[int, int]:
    for block in schema["blocks"]:
        if block["block"] == block_name:
            return int(block["start"]), int(block["end_exclusive"])
    raise KeyError(f"Block '{block_name}' missing from schema")


@dataclass(frozen=True)
class BranchSpec:
    name: str
    root: Path
    width: int
    face_start: int
    face_end: int
    mask_start: int
    mask_end: int
    face_landmarks: int

    @property
    def tail_start(self) -> int:
        return self.face_end


def build_spec(name: str, root: Path, schema: Dict[str, Any]) -> BranchSpec:
    face_start, face_end = block_bounds(schema, "face")
    mask_start, mask_end = block_bounds(schema, "presence_mask")
    return BranchSpec(name=name, root=root, width=int(schema["features_per_frame"]),
                      face_start=face_start, face_end=face_end,
                      mask_start=mask_start, mask_end=mask_end,
                      face_landmarks=int(schema["face_landmarks_kept"]))


FULL = build_spec("full", FULL_DIR, FULL_SCHEMA)
REDUCED = build_spec("reduced", REDUCED_DIR, REDUCED_SCHEMA)
PIPELINE_ID = FULL_SCHEMA["pipeline_id"]

FULL_TAIL_WIDTH = FULL.width - FULL.tail_start
REDUCED_TAIL_WIDTH = REDUCED.width - REDUCED.tail_start
print(f"\npipeline id       : {PIPELINE_ID}")
print(f"shared tail width : full={FULL_TAIL_WIDTH}, reduced={REDUCED_TAIL_WIDTH}")
if FULL_TAIL_WIDTH != REDUCED_TAIL_WIDTH:
    raise RuntimeError("Shared tail widths differ; the branches are not comparable.")

# ---- Negative self-tests for this cell -----------------------------------
_cell2_failures: List[str] = []


def _expect(name: str, condition: bool, detail: str = "") -> None:
    print(f"{'PASS' if condition else 'FAIL'}  {name}  {detail}")
    if not condition:
        _cell2_failures.append(name)


print("\nNegative self-tests")
_bad_width = json.loads(json.dumps(REDUCED_SCHEMA))
_bad_width["features_per_frame"] = 630
_expect("wrong features_per_frame rejected", bool(validate_schema("reduced", _bad_width)))

_bad_boundary = json.loads(json.dumps(REDUCED_SCHEMA))
_bad_boundary["blocks"][1]["start"] = 275          # creates an overlap
_expect("shifted block boundary rejected", bool(validate_schema("reduced", _bad_boundary)))

_bad_name = json.loads(json.dumps(REDUCED_SCHEMA))
_bad_name["blocks"][2]["block"] = "left_hand_wrong"
_expect("renamed block rejected", bool(validate_schema("reduced", _bad_name)))

_bad_mask = json.loads(json.dumps(FULL_SCHEMA))
_bad_mask["blocks"][-1]["end_exclusive"] = 1758
_expect("wrong presence-mask width rejected", bool(validate_schema("full", _bad_mask)))

_bad_pipeline = json.loads(json.dumps(FULL_SCHEMA))
_bad_pipeline["pipeline_id"] = "deadbeefcafe"
_expect("wrong pipeline id rejected", bool(validate_schema("full", _bad_pipeline)))

_expect("correct schemas accepted",
        not validate_schema("full", FULL_SCHEMA) and not validate_schema("reduced", REDUCED_SCHEMA))

_root_a, _root_b = Path("/tmp/_bundle_a"), Path("/tmp/_bundle_b")
_expect("override from mismatched roots rejected",
        bool(check_override_coherence(
            "full", _root_a / "normalized_landmarks" / "full",
            _root_b / "normalized_metadata_full.csv",
            _root_a / "normalized_feature_schema_full.json")))
_expect("override from one root accepted",
        not check_override_coherence(
            "full", _root_a / "normalized_landmarks" / "full",
            _root_a / "normalized_metadata_full.csv",
            _root_a / "normalized_feature_schema_full.json"))
_expect("override with wrong directory shape rejected",
        bool(check_override_coherence("full", _root_a / "landmarks" / "full",
                                      _root_a / "normalized_metadata_full.csv",
                                      _root_a / "normalized_feature_schema_full.json")))

if _cell2_failures:
    raise RuntimeError(f"Cell 2 self-tests failed: {_cell2_failures}")
print("\nCell 2 self-tests clean.")

In [ ]:
# ============================================================
# CELL 3 — Disk inventory, metadata comparison, canonical clip hash
# ============================================================
# Deliberately does NOT start from the metadata: listing the directories
# independently is the only way to catch orphan .npy files on disk, or
# metadata rows pointing at files that are not there.

full_files = sorted(FULL_DIR.rglob("*.npy"))
reduced_files = sorted(REDUCED_DIR.rglob("*.npy"))
full_disk = [p.relative_to(FULL_DIR).as_posix() for p in full_files]
reduced_disk = [p.relative_to(REDUCED_DIR).as_posix() for p in reduced_files]
full_disk_set, reduced_disk_set = set(full_disk), set(reduced_disk)

print("FILE INVENTORY (independent of metadata)")
print(f"  .npy on disk : full={len(full_files):,}, reduced={len(reduced_files):,}")
print(f"  duplicate paths: full={len(full_disk) - len(full_disk_set)}, "
      f"reduced={len(reduced_disk) - len(reduced_disk_set)}")

disk_only_full = sorted(full_disk_set - reduced_disk_set)
disk_only_reduced = sorted(reduced_disk_set - full_disk_set)
print(f"  on full not reduced: {len(disk_only_full):,} | "
      f"on reduced not full: {len(disk_only_reduced):,}")

full_meta = pd.read_csv(FULL_METADATA_PATH)
reduced_meta = pd.read_csv(REDUCED_METADATA_PATH)
print(f"\nmetadata rows: full={len(full_meta):,}, reduced={len(reduced_meta):,}")


def normalise_path(value: Any) -> str:
    return Path(str(value).replace("\\", "/")).as_posix()


full_meta["_relative"] = full_meta["relative_path"].map(normalise_path)
reduced_meta["_relative"] = reduced_meta["relative_path"].map(normalise_path)

orphans_full = sorted(full_disk_set - set(full_meta["_relative"]))
orphans_reduced = sorted(reduced_disk_set - set(reduced_meta["_relative"]))
absent_full = sorted(set(full_meta["_relative"]) - full_disk_set)
absent_reduced = sorted(set(reduced_meta["_relative"]) - reduced_disk_set)

print("\nDISK vs METADATA")
print(f"  orphan (on disk, not in metadata): full={len(orphans_full):,}, "
      f"reduced={len(orphans_reduced):,}")
print(f"  absent (in metadata, not on disk): full={len(absent_full):,}, "
      f"reduced={len(absent_reduced):,}")
for name, listing in (("orphan full", orphans_full), ("orphan reduced", orphans_reduced),
                      ("absent full", absent_full), ("absent reduced", absent_reduced)):
    for entry in listing[:5]:
        print(f"    {name}: {entry}")

duplicates_full = full_meta["clip"][full_meta["clip"].duplicated()].tolist()
duplicates_reduced = reduced_meta["clip"][reduced_meta["clip"].duplicated()].tolist()
duplicate_paths_full = full_meta["_relative"][full_meta["_relative"].duplicated()].tolist()
duplicate_paths_reduced = reduced_meta["_relative"][reduced_meta["_relative"].duplicated()].tolist()
print(f"\nduplicate clip ids  : full={len(duplicates_full)}, reduced={len(duplicates_reduced)}")
print(f"duplicate rel paths : full={len(duplicate_paths_full)}, "
      f"reduced={len(duplicate_paths_reduced)}")

# ---- Compare every shared metadata column --------------------------------
BRANCH_SPECIFIC = {"variant", "normalized_file", "features", "_relative"}
shared_columns = sorted((set(full_meta.columns) & set(reduced_meta.columns)) - BRANCH_SPECIFIC)
merged = full_meta.merge(reduced_meta, on="clip", how="outer",
                         suffixes=("_full", "_reduced"), indicator=True)
only_full = merged.loc[merged["_merge"] == "left_only", "clip"].tolist()
only_reduced = merged.loc[merged["_merge"] == "right_only", "clip"].tolist()
both = merged[merged["_merge"] == "both"]


def column_disagreements(column: str) -> pd.DataFrame:
    left, right = f"{column}_full", f"{column}_reduced"
    if left not in both.columns or right not in both.columns:
        return pd.DataFrame()
    left_values, right_values = both[left], both[right]
    if pd.api.types.is_numeric_dtype(left_values) and pd.api.types.is_numeric_dtype(right_values):
        differing = ~np.isclose(left_values.astype(float), right_values.astype(float),
                                rtol=0, atol=0, equal_nan=True)
    else:
        differing = left_values.astype(str) != right_values.astype(str)
    return both.loc[differing, ["clip", left, right]]


metadata_disagreements = {column: column_disagreements(column) for column in shared_columns}
total_disagreements = sum(len(frame) for frame in metadata_disagreements.values())
print(f"\nshared metadata columns compared: {len(shared_columns)}")
print(f"  {shared_columns}")
print(f"  excluded as branch-specific: {sorted(BRANCH_SPECIFIC - {'_relative'})}")
print(f"  disagreements: {total_disagreements:,}")
for column, frame in metadata_disagreements.items():
    if len(frame):
        print(f"    {column}: {len(frame):,}  <-- MISMATCH")

# ---- Canonical clip table ------------------------------------------------
clips = full_meta[full_meta["clip"].isin(both["clip"])].copy()
clips = clips.drop(columns=["relative_path"]).rename(
    columns={"frames": "meta_frames", "_relative": "relative_path"})
for part in ("face", "pose", "left_hand", "right_hand"):
    if f"{part}_detection_rate" in clips.columns:
        clips[f"meta_{part}_rate"] = clips[f"{part}_detection_rate"]
keep = ["clip", "label", "signer_id", "session", "relative_path", "meta_frames"]
clips = clips[[c for c in keep if c in clips.columns]
              + [c for c in clips.columns if c.startswith("meta_") and c.endswith("_rate")]]
clips = clips.sort_values("clip", kind="mergesort").reset_index(drop=True)

# ---- Canonical content hash ----------------------------------------------
HASH_COLUMNS = ["clip", "label", "signer_id", "session", "relative_path", "meta_frames",
                "meta_face_rate", "meta_pose_rate", "meta_left_hand_rate",
                "meta_right_hand_rate"]


def canonical_value(column: str, value: Any) -> str:
    """One stable text form per field, so the hash is reproducible."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "nan"
    if column == "relative_path":
        return normalise_path(value)
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.10g}"
    if isinstance(value, (int, np.integer, np.bool_, bool)):
        return str(int(value))
    return str(value)


def canonical_clip_hash(frame: pd.DataFrame) -> Tuple[str, List[str]]:
    """Content hash over clip identity, labels, paths and frame counts.

    Hashing clip ids alone would give the same value to a dataset whose labels,
    signers or paths had changed, which is exactly what must not happen.
    """
    columns = [c for c in HASH_COLUMNS if c in frame.columns]
    ordered = frame.sort_values("clip", kind="mergesort")[columns]
    lines = []
    for record in ordered.to_dict("records"):
        lines.append("\t".join(f"{column}={canonical_value(column, record[column])}"
                               for column in columns))
    payload = "\n".join(lines) + "\n"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16], columns


CLIP_TABLE_HASH, HASHED_COLUMNS = canonical_clip_hash(clips)

print(f"\ncanonical clip table : {len(clips):,} rows")
print(f"  labels {clips['label'].nunique()} | signers {clips['signer_id'].nunique()} | "
      f"unique clip ids {clips['clip'].nunique()}")
print(f"  hashed columns : {HASHED_COLUMNS}")
print(f"  clip table hash: {CLIP_TABLE_HASH}")

coverage = clips.groupby("signer_id")["label"].nunique()
incomplete_signers = coverage[coverage < EXPECTED_LABELS]
print(f"\nclass coverage per signer: min={coverage.min()}, max={coverage.max()}")
if len(incomplete_signers):
    print(incomplete_signers.to_string())
else:
    print(f"  every signer covers all {EXPECTED_LABELS} classes")

# ---- Hash sensitivity self-tests -----------------------------------------
_cell3_failures: List[str] = []


def _expect3(name: str, condition: bool, detail: str = "") -> None:
    print(f"{'PASS' if condition else 'FAIL'}  {name}  {detail}")
    if not condition:
        _cell3_failures.append(name)


print("\nCanonical hash self-tests")
_base = pd.DataFrame({
    "clip": ["p1_c_aam", "p2_f_boi"], "label": ["aam", "boi"],
    "signer_id": ["p1", "p2"], "session": ["c", "f"],
    "relative_path": ["DATASET/aam/p1_c_aam.npy", "DATASET/boi/p2_f_boi.npy"],
    "meta_frames": [100, 120], "meta_face_rate": [1.0, 0.5],
})
_base_hash, _ = canonical_clip_hash(_base)

for _field, _new in (("label", "different"), ("signer_id", "p9"),
                     ("meta_frames", 999), ("relative_path", "DATASET/x/y.npy"),
                     ("session", "z"), ("meta_face_rate", 0.25)):
    _changed = _base.copy()
    _changed.loc[0, _field] = _new
    _expect3(f"changing {_field} changes the hash",
             canonical_clip_hash(_changed)[0] != _base_hash)

_reordered = _base.iloc[::-1].reset_index(drop=True)
_expect3("row order does not change the hash", canonical_clip_hash(_reordered)[0] == _base_hash)

_windows = _base.copy()
_windows.loc[0, "relative_path"] = "DATASET\\aam\\p1_c_aam.npy"
_expect3("path separators are normalised", canonical_clip_hash(_windows)[0] == _base_hash)

_expect3("identical tables hash identically",
         canonical_clip_hash(_base.copy())[0] == _base_hash, _base_hash)

if _cell3_failures:
    raise RuntimeError(f"Cell 3 self-tests failed: {_cell3_failures}")
print("\nCell 3 self-tests clean.")

In [ ]:
# ============================================================
# CELL 4 — Audit functions, enrichment, and self-tests
# ============================================================

def audit_arrays(full_array: np.ndarray, reduced_array: np.ndarray,
                 full_spec: BranchSpec, reduced_spec: BranchSpec) -> Dict[str, Any]:
    record: Dict[str, Any] = {
        "full_frames": int(full_array.shape[0]),
        "reduced_frames": int(reduced_array.shape[0]),
        "full_width": int(full_array.shape[1]),
        "reduced_width": int(reduced_array.shape[1]),
    }
    record["frames_match"] = record["full_frames"] == record["reduced_frames"]
    record["full_width_ok"] = record["full_width"] == full_spec.width
    record["reduced_width_ok"] = record["reduced_width"] == reduced_spec.width
    record["full_finite"] = bool(np.isfinite(full_array).all())
    record["reduced_finite"] = bool(np.isfinite(reduced_array).all())

    full_mask = full_array[:, full_spec.mask_start:full_spec.mask_end]
    reduced_mask = reduced_array[:, reduced_spec.mask_start:reduced_spec.mask_end]
    record["full_mask_binary"] = bool(np.isin(full_mask, (0.0, 1.0)).all())
    record["reduced_mask_binary"] = bool(np.isin(reduced_mask, (0.0, 1.0)).all())
    record["masks_identical"] = bool(full_mask.shape == reduced_mask.shape
                                     and np.array_equal(full_mask, reduced_mask))
    for index, part in enumerate(("face", "pose", "left_hand", "right_hand")):
        record[f"computed_{part}_rate"] = float(full_mask[:, index].mean())

    # Two distinct claims. Numeric equality treats +0.0 and -0.0 as equal;
    # byte equality does not. Numeric equality is what matters scientifically;
    # byte equality is the stronger statement. Both are recorded.
    full_tail = full_array[:, full_spec.tail_start:]
    reduced_tail = reduced_array[:, reduced_spec.tail_start:]
    if full_tail.shape == reduced_tail.shape:
        difference = float(np.abs(full_tail - reduced_tail).max()) if full_tail.size else 0.0
        record["tail_shape_match"] = True
        record["tail_max_abs_diff"] = difference
        record["tail_numerically_identical"] = difference == 0.0
        record["tail_byte_identical"] = bool(full_tail.tobytes() == reduced_tail.tobytes())
    else:
        record["tail_shape_match"] = False
        record["tail_max_abs_diff"] = float("nan")
        record["tail_numerically_identical"] = False
        record["tail_byte_identical"] = False

    record["full_min"] = float(full_array.min())
    record["full_max"] = float(full_array.max())
    record["reduced_min"] = float(reduced_array.min())
    record["reduced_max"] = float(reduced_array.max())
    record["over_threshold"] = record["full_frames"] > LONG_CLIP_THRESHOLD
    return record


def clip_sufficient_statistics(array: np.ndarray) -> Dict[str, Any]:
    """Frame count, column sums and sums of squares, in float64.

    These three quantities ADD, so summing them over any set of signers gives
    exactly the mean and standard deviation you would get from loading those
    signers directly. float64 matters: ~800k frames in float32 loses precision.
    """
    wide = array.astype(np.float64, copy=False)
    return {"frames": int(wide.shape[0]), "sum": wide.sum(axis=0),
            "sumsq": np.square(wide).sum(axis=0)}


def audit_clip(task: Dict[str, str]) -> Dict[str, Any]:
    """Read one clip from both branches and audit it. Never raises."""
    record: Dict[str, Any] = {"clip": task["clip"], "relative_path": task["relative_path"]}
    try:
        full_array = np.load(task["full_path"])
        reduced_array = np.load(task["reduced_path"])
        record.update(audit_arrays(full_array, reduced_array, FULL, REDUCED))
        record["_statistics"] = {"signer": task["signer_id"],
                                 "full": clip_sufficient_statistics(full_array),
                                 "reduced": clip_sufficient_statistics(reduced_array)}
        record["status"] = "ok"
    except Exception as error:
        record["status"] = "failed"
        record["error_type"] = type(error).__name__
        record["error_message"] = str(error)
    return record


ENRICHMENT_COLUMNS = ["label", "signer_id", "session", "meta_frames"]


def enrich_audit(raw: pd.DataFrame, clip_table: pd.DataFrame) -> pd.DataFrame:
    """Attach metadata to raw audit records.

    Raw records deliberately carry no metadata columns, so this is safe to run
    on a resumed checkpoint. Any that somehow appear are dropped first, because
    merging a column onto itself yields label_x/label_y and every later lookup
    of the plain name then fails.
    """
    columns = [c for c in ENRICHMENT_COLUMNS if c in clip_table.columns]
    columns += [c for c in clip_table.columns
                if c.startswith("meta_") and c.endswith("_rate")]
    raw = raw.drop(columns=[c for c in columns if c in raw.columns], errors="ignore")
    merged = raw.merge(clip_table[["clip"] + columns], on="clip", how="left")
    suffixed = [c for c in merged.columns if c.endswith("_x") or c.endswith("_y")]
    if suffixed:
        raise RuntimeError(f"Enrichment produced duplicated columns: {suffixed}")
    return merged


RAW_RECORD_DROP = ("_statistics",)


def to_raw_record(record: Dict[str, Any]) -> Dict[str, Any]:
    return {key: value for key, value in record.items() if key not in RAW_RECORD_DROP}


# ---- Self-tests -----------------------------------------------------------
_failures: List[str] = []


def _check(name: str, condition: bool, detail: str = "") -> None:
    print(f"{'PASS' if condition else 'FAIL'}  {name}  {detail}")
    if not condition:
        _failures.append(name)


def _synthetic(frames: int, spec: BranchSpec, tail_seed: int, face_seed: int) -> np.ndarray:
    array = np.zeros((frames, spec.width), dtype=np.float32)
    array[:, spec.face_start:spec.face_end] = np.random.default_rng(face_seed).normal(
        0, 1, size=(frames, spec.face_end - spec.face_start))
    array[:, spec.tail_start:spec.mask_start] = np.random.default_rng(tail_seed).normal(
        0, 1, size=(frames, spec.width - spec.tail_start - 4))
    array[:, spec.mask_start:spec.mask_end] = (
        np.random.default_rng(tail_seed + 1).random((frames, 4)) > 0.3).astype(np.float32)
    return array


good_full = _synthetic(40, FULL, tail_seed=11, face_seed=1)
good_reduced = _synthetic(40, REDUCED, tail_seed=11, face_seed=2)
result = audit_arrays(good_full, good_reduced, FULL, REDUCED)
_check("aligned branches: tails numerically identical", result["tail_numerically_identical"],
       f"max diff {result['tail_max_abs_diff']:.1e}")
_check("aligned branches: tails byte identical", result["tail_byte_identical"])
_check("masks recognised as binary",
       result["full_mask_binary"] and result["reduced_mask_binary"])
_check("masks identical across branches", result["masks_identical"])
_check("widths validated", result["full_width_ok"] and result["reduced_width_ok"])

_bad = good_reduced.copy()
_bad[3, REDUCED.tail_start + 5] += 0.01
_corrupted = audit_arrays(good_full, _bad, FULL, REDUCED)
_check("a corrupted tail is detected",
       not _corrupted["tail_numerically_identical"] and not _corrupted["tail_byte_identical"])

_negative = good_reduced.copy()
_negative[0, REDUCED.tail_start + 1] = -0.0
_zeroed = good_full.copy()
_zeroed[0, FULL.tail_start + 1] = 0.0
_nz = audit_arrays(_zeroed, _negative, FULL, REDUCED)
_check("negative zero is numerically equal", _nz["tail_numerically_identical"])
_check("negative zero differs at byte level", not _nz["tail_byte_identical"])

_fractional = good_full.copy()
_fractional[2, FULL.mask_start] = 0.5
_check("a non-binary mask is detected",
       not audit_arrays(_fractional, good_reduced, FULL, REDUCED)["full_mask_binary"])

_nan = good_full.copy(); _nan[1, 0] = np.nan
_check("a NaN is detected", not audit_arrays(_nan, good_reduced, FULL, REDUCED)["full_finite"])
_inf = good_full.copy(); _inf[1, 0] = np.inf
_check("an Inf is detected", not audit_arrays(_inf, good_reduced, FULL, REDUCED)["full_finite"])

_check("long clips flagged",
       audit_arrays(_synthetic(300, FULL, 5, 1), _synthetic(300, REDUCED, 5, 2),
                    FULL, REDUCED)["over_threshold"])
_check("frame-count mismatch detected",
       not audit_arrays(good_full, _synthetic(41, REDUCED, 11, 2), FULL, REDUCED)["frames_match"])

_a, _b = _synthetic(37, REDUCED, 3, 4), _synthetic(51, REDUCED, 6, 7)
_sa, _sb = clip_sufficient_statistics(_a), clip_sufficient_statistics(_b)
_frames = _sa["frames"] + _sb["frames"]
_mean = (_sa["sum"] + _sb["sum"]) / _frames
_std = np.sqrt(np.maximum((_sa["sumsq"] + _sb["sumsq"]) / _frames - _mean ** 2, 0.0))
_direct = np.concatenate([_a, _b], axis=0).astype(np.float64)
_check("combined sums reproduce the true mean",
       np.abs(_mean - _direct.mean(axis=0)).max() < 1e-9,
       f"max diff {np.abs(_mean - _direct.mean(axis=0)).max():.2e}")
_check("combined sums reproduce the true std",
       np.abs(_std - _direct.std(axis=0)).max() < 1e-8,
       f"max diff {np.abs(_std - _direct.std(axis=0)).max():.2e}")

_check("missing file reported, not raised",
       audit_clip({"clip": "x", "relative_path": "x", "signer_id": "p0",
                   "full_path": "/nonexistent.npy",
                   "reduced_path": "/nonexistent.npy"})["status"] == "failed")

# ---- The v3 resume bug, tested directly ----------------------------------
_raw = pd.DataFrame([{"clip": "p1_c_aam", "status": "ok", "full_frames": 100},
                     {"clip": "p2_f_boi", "status": "ok", "full_frames": 120}])
_clip_table = pd.DataFrame({"clip": ["p1_c_aam", "p2_f_boi"], "label": ["aam", "boi"],
                            "signer_id": ["p1", "p2"], "session": ["c", "f"],
                            "meta_frames": [100, 120], "meta_face_rate": [1.0, 0.9]})
_first = enrich_audit(_raw, _clip_table)
_check("enrichment yields one signer_id column",
       list(_first.columns).count("signer_id") == 1)
_check("enrichment yields no _x/_y columns",
       not [c for c in _first.columns if c.endswith(("_x", "_y"))])

# Enriching a SECOND time from the same raw records - the exact resume path.
_second = enrich_audit(_raw, _clip_table)
_check("re-enriching raw records stays clean",
       list(_second.columns).count("signer_id") == 1
       and not [c for c in _second.columns if c.endswith(("_x", "_y"))])

# And enriching an already-enriched frame must not duplicate either.
_third = enrich_audit(_first, _clip_table)
_check("enriching an enriched frame stays clean",
       list(_third.columns).count("signer_id") == 1
       and _third["signer_id"].tolist() == ["p1", "p2"])

_check("raw records exclude the statistics payload",
       "_statistics" not in to_raw_record({"clip": "x", "_statistics": {}, "status": "ok"}))

print()
if _failures:
    raise RuntimeError(f"Self-tests failed: {_failures}")
print("Self-tests clean.")

In [ ]:
# ============================================================
# CELL 5 — Run the audit (raw checkpoint), then enrich
# ============================================================
tasks: List[Dict[str, str]] = []
missing_files: List[Dict[str, Any]] = []

for row in clips.itertuples(index=False):
    relative = Path(str(row.relative_path))
    full_path, reduced_path = FULL.root / relative, REDUCED.root / relative
    if not full_path.exists() or not reduced_path.exists():
        missing_files.append({"clip": row.clip, "full_exists": full_path.exists(),
                              "reduced_exists": reduced_path.exists()})
        continue
    tasks.append({"clip": row.clip, "relative_path": relative.as_posix(),
                  "signer_id": str(row.signer_id),
                  "full_path": str(full_path), "reduced_path": str(reduced_path)})

print(f"clips to audit  : {len(tasks):,}")
print(f"missing on disk : {len(missing_files):,}")

if SMOKE_TEST and len(tasks) > SMOKE_TEST_CLIPS:
    tasks = tasks[::max(1, len(tasks) // SMOKE_TEST_CLIPS)][:SMOKE_TEST_CLIPS]
    print(f"SMOKE TEST: reduced to {len(tasks):,} evenly spaced clips")

EXPECTED_AUDIT_COUNT = len(tasks) if SMOKE_TEST else EXPECTED_CLIPS

accumulators: Dict[str, Dict[str, Dict[str, Any]]] = {"full": {}, "reduced": {}}
processed_clips: set = set()


def accumulate(statistics: Dict[str, Any]) -> None:
    signer = statistics["signer"]
    for branch in ("full", "reduced"):
        clip_statistics = statistics[branch]
        bucket = accumulators[branch].get(signer)
        if bucket is None:
            accumulators[branch][signer] = {"frames": clip_statistics["frames"],
                                            "sum": clip_statistics["sum"].copy(),
                                            "sumsq": clip_statistics["sumsq"].copy()}
        else:
            bucket["frames"] += clip_statistics["frames"]
            bucket["sum"] += clip_statistics["sum"]
            bucket["sumsq"] += clip_statistics["sumsq"]


def save_checkpoint(records: List[Dict[str, Any]]) -> None:
    """Raw records and accumulators, both atomic, clip ids stored in the npz."""
    atomic_write(RAW_CHECKPOINT,
                 lambda handle: pd.DataFrame(records).to_csv(handle, index=False))
    payload: Dict[str, np.ndarray] = {"__clips__": np.array(sorted(processed_clips))}
    for branch, per_signer in accumulators.items():
        for signer, bucket in per_signer.items():
            payload[f"{branch}|{signer}|frames"] = np.array([bucket["frames"]], dtype=np.int64)
            payload[f"{branch}|{signer}|sum"] = bucket["sum"]
            payload[f"{branch}|{signer}|sumsq"] = bucket["sumsq"]
    atomic_write(ACCUMULATOR_CHECKPOINT,
                 lambda handle: np.savez_compressed(handle, **payload))


def load_accumulators(path: Path) -> set:
    restored: set = set()
    with np.load(path, allow_pickle=False) as archive:
        for key in archive.files:
            if key == "__clips__":
                restored = {str(value) for value in archive[key]}
                continue
            branch, signer, field = key.split("|")
            bucket = accumulators[branch].setdefault(
                signer, {"frames": 0, "sum": None, "sumsq": None})
            if field == "frames":
                bucket["frames"] = int(archive[key][0])
            else:
                bucket[field] = archive[key]
    return restored


completed: List[Dict[str, Any]] = []
if RAW_CHECKPOINT.exists() and not OVERWRITE:
    previous = pd.read_csv(RAW_CHECKPOINT)
    done = set(previous["clip"].astype(str))
    remaining = [t for t in tasks if t["clip"] not in done]
    if ACCUMULATOR_CHECKPOINT.exists():
        restored = load_accumulators(ACCUMULATOR_CHECKPOINT)
        if restored == done:
            completed = previous.to_dict("records")
            processed_clips.update(restored)
            tasks = remaining
            print(f"Resuming: {len(completed):,} audited, {len(tasks):,} remaining")
        else:
            print(f"Checkpoint disagreement (csv {len(done):,} vs npz {len(restored):,}). "
                  "Restarting the pass.")
            accumulators = {"full": {}, "reduced": {}}
    else:
        print("Raw checkpoint found but no statistics checkpoint. Restarting the pass.")
elif OVERWRITE:
    print("OVERWRITE=True: ignoring any existing checkpoint.")

started = time.time()
if tasks:
    with multiprocessing.get_context("fork").Pool(NUM_WORKERS) as pool:
        stream = pool.imap_unordered(audit_clip, tasks, chunksize=8)
        for index, record in enumerate(
                tqdm(stream, total=len(tasks), desc="Auditing", unit="clip",
                     mininterval=10.0), 1):
            statistics = record.pop("_statistics", None)
            if statistics is not None:
                accumulate(statistics)
                processed_clips.add(record["clip"])
            completed.append(to_raw_record(record))
            if index % CHECKPOINT_EVERY == 0:
                save_checkpoint(completed)
save_checkpoint(completed)

raw_audit = pd.DataFrame(completed).sort_values("clip", kind="mergesort").reset_index(drop=True)
audit = enrich_audit(raw_audit, clips)
atomic_write(AUDIT_CSV, lambda handle: audit.to_csv(handle, index=False))

ok = audit[audit["status"] == "ok"].copy()
failed_reads = audit[audit["status"] != "ok"]

print(f"\nAudited {len(audit):,} clips in {(time.time() - started) / 60:.1f} min")
print(f"Read failures: {len(failed_reads):,}")
if not failed_reads.empty:
    print(failed_reads[["clip", "error_type", "error_message"]].head(10).to_string())
print(f"raw checkpoint : {RAW_CHECKPOINT.name}")
print(f"final audit    : {AUDIT_CSV.name}")

In [ ]:
# ============================================================
# CELL 6 — Per-signer sufficient statistics (unverified until PASS)
# ============================================================
# NOT a fitted transformation: raw per-signer sums. Notebook 8 adds only the 12
# TRAINING signers of a fold, so validation and test signers cannot influence
# the scaling. Written to _unverified_* here; published only after the verdict.

expected_signers = set(clips["signer_id"].astype(str))
statistics_complete = True
statistics_report: Dict[str, Any] = {}
STATS_PAYLOADS: Dict[str, Dict[str, np.ndarray]] = {}

audited_frames = int(ok["full_frames"].sum())

for branch, spec in (("full", FULL), ("reduced", REDUCED)):
    per_signer = accumulators[branch]
    missing = expected_signers - set(per_signer)
    frames_total = sum(bucket["frames"] for bucket in per_signer.values())
    widths_ok = all(bucket["sum"].shape == (spec.width,) for bucket in per_signer.values())
    finite_ok = all(np.isfinite(bucket["sum"]).all() and np.isfinite(bucket["sumsq"]).all()
                    for bucket in per_signer.values())
    frames_ok = frames_total == audited_frames and not missing

    statistics_report[branch] = {
        "signers": len(per_signer), "missing_signers": sorted(missing),
        "total_frames": int(frames_total), "audited_frames": audited_frames,
        "frames_match": bool(frames_ok), "widths_ok": bool(widths_ok),
        "finite": bool(finite_ok),
    }
    statistics_complete &= bool(frames_ok and widths_ok and finite_ok)

    payload: Dict[str, np.ndarray] = {}
    for signer in sorted(per_signer):
        bucket = per_signer[signer]
        payload[f"{signer}|frames"] = np.array([bucket["frames"]], dtype=np.int64)
        payload[f"{signer}|sum"] = bucket["sum"]
        payload[f"{signer}|sumsq"] = bucket["sumsq"]
    payload.update({
        "__branch__": np.array([branch]),
        "__audit_version__": np.array([AUDIT_VERSION]),
        "__pipeline_id__": np.array([PIPELINE_ID]),
        "__clip_table_hash__": np.array([CLIP_TABLE_HASH]),
        "__metadata_hash__": np.array([METADATA_HASHES[branch]]),
        "__schema_hash__": np.array([SCHEMA_HASHES[branch]]),
        "__block_layout_hash__": np.array([BLOCK_LAYOUT_HASHES[branch]]),
        "__signers__": np.array(sorted(per_signer)),
        "__width__": np.array([spec.width], dtype=np.int64),
        "__tail_start__": np.array([spec.tail_start], dtype=np.int64),
        "__mask_columns__": np.arange(spec.mask_start, spec.mask_end, dtype=np.int64),
        "__total_frames__": np.array([frames_total], dtype=np.int64),
    })
    STATS_PAYLOADS[branch] = payload
    atomic_write(UNVERIFIED_STATS[branch],
                 lambda handle, p=payload: np.savez_compressed(handle, **p))

print("PER-SIGNER SUFFICIENT STATISTICS (unverified)")
for branch, report in statistics_report.items():
    print(f"  {branch:<8} signers={report['signers']:<3} frames={report['total_frames']:,} "
          f"(audited {report['audited_frames']:,}) widths_ok={report['widths_ok']} "
          f"finite={report['finite']}")
    if report["missing_signers"]:
        print(f"           MISSING: {report['missing_signers']}")

# The audit proved the saved arrays align. This proves the standardisation
# INPUTS derived from them also align, which is what notebook 8 consumes.
tail_statistics_aligned = True
tail_statistics_detail: Dict[str, Any] = {}
for signer in sorted(expected_signers):
    full_bucket = accumulators["full"].get(signer)
    reduced_bucket = accumulators["reduced"].get(signer)
    if full_bucket is None or reduced_bucket is None:
        tail_statistics_aligned = False
        tail_statistics_detail[signer] = "missing from one branch"
        continue
    if not (full_bucket["frames"] == reduced_bucket["frames"]
            and np.array_equal(full_bucket["sum"][FULL.tail_start:],
                               reduced_bucket["sum"][REDUCED.tail_start:])
            and np.array_equal(full_bucket["sumsq"][FULL.tail_start:],
                               reduced_bucket["sumsq"][REDUCED.tail_start:])):
        tail_statistics_aligned = False
        tail_statistics_detail[signer] = "shared-tail statistics differ"

print(f"\nshared-tail statistics aligned across branches: {tail_statistics_aligned}")
for signer, detail in tail_statistics_detail.items():
    print(f"    {signer}: {detail}")


def fold_statistics(branch: str, training_signers: Sequence[str],
                    mask_columns: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Mean and std from training signers only. This is what notebook 8 calls."""
    buckets = [accumulators[branch][signer] for signer in training_signers]
    frames = sum(bucket["frames"] for bucket in buckets)
    mean = np.sum([b["sum"] for b in buckets], axis=0) / frames
    variance = np.sum([b["sumsq"] for b in buckets], axis=0) / frames - mean ** 2
    std = np.sqrt(np.maximum(variance, 0.0))
    std = np.where(std < 1e-8, 1.0, std)
    mean[mask_columns] = 0.0
    std[mask_columns] = 1.0
    return mean, std


_demo_signers = sorted(expected_signers)[:max(1, len(expected_signers) - 4)]
_demo_mean, _demo_std = fold_statistics("reduced", _demo_signers,
                                        np.arange(REDUCED.mask_start, REDUCED.mask_end))
print(f"\n  demo fold ({len(_demo_signers)} training signers): "
      f"mean {_demo_mean.min():.3f}..{_demo_mean.max():.3f}, "
      f"std {_demo_std.min():.3f}..{_demo_std.max():.3f}")
print(f"  mask columns untouched: "
      f"{bool(np.all(_demo_mean[REDUCED.mask_start:] == 0.0) and np.all(_demo_std[REDUCED.mask_start:] == 1.0))}")
print(f"\n  unverified files written: "
      f"{UNVERIFIED_STATS['full'].name}, {UNVERIFIED_STATS['reduced'].name}")

In [ ]:
# ============================================================
# CELL 7 — Aggregate: distributions, per-signer quality, alignment
# ============================================================
lengths = ok["full_frames"].to_numpy()
length_stats = {
    "count": int(len(lengths)), "min": int(lengths.min()), "max": int(lengths.max()),
    "mean": float(lengths.mean()), "median": float(np.median(lengths)),
    "std": float(lengths.std()), "p90": float(np.percentile(lengths, 90)),
    "p95": float(np.percentile(lengths, 95)), "p99": float(np.percentile(lengths, 99)),
    "total_frames": int(lengths.sum()),
    f"over_{LONG_CLIP_THRESHOLD}_count": int((lengths > LONG_CLIP_THRESHOLD).sum()),
    f"over_{LONG_CLIP_THRESHOLD}_percent": float(100 * (lengths > LONG_CLIP_THRESHOLD).mean()),
}
print("SEQUENCE LENGTH")
for key, value in length_stats.items():
    print(f"  {key}: {value:,.2f}" if isinstance(value, float) else f"  {key}: {value:,}")

signer_summary = ok.groupby("signer_id").agg(
    clips=("clip", "size"), classes=("label", "nunique"),
    mean_frames=("full_frames", "mean"), median_frames=("full_frames", "median"),
    min_frames=("full_frames", "min"), max_frames=("full_frames", "max"),
    total_frames=("full_frames", "sum"), clips_over_threshold=("over_threshold", "sum"),
    face_rate=("computed_face_rate", "mean"), pose_rate=("computed_pose_rate", "mean"),
    left_hand_rate=("computed_left_hand_rate", "mean"),
    right_hand_rate=("computed_right_hand_rate", "mean"),
).round(4)
signer_summary["clips_over_threshold_pct"] = (
    100 * signer_summary["clips_over_threshold"] / signer_summary["clips"]).round(2)
signer_summary = signer_summary.sort_values("left_hand_rate")
atomic_write(SIGNER_CSV, lambda handle: signer_summary.to_csv(handle))
print("\nPER-SIGNER SUMMARY")
print(signer_summary.to_string())

rate_agreement = {}
for part in ("face", "pose", "left_hand", "right_hand"):
    meta_column = f"meta_{part}_rate"
    if meta_column in ok.columns:
        difference = (ok[meta_column] - ok[f"computed_{part}_rate"]).abs()
        rate_agreement[part] = {
            "max_abs_difference": float(difference.max()),
            "mean_abs_difference": float(difference.mean()),
            "clips_differing_over_1e-6": int((difference > 1e-6).sum())}

print("\nDETECTION RATES: metadata vs recomputed from masks")
for part, stats in rate_agreement.items():
    print(f"  {part:<12} max diff {stats['max_abs_difference']:.2e} | "
          f"differing: {stats['clips_differing_over_1e-6']:,}")

aggregate_rates = {part: {
    "mean": float(ok[f"computed_{part}_rate"].mean()),
    "median": float(ok[f"computed_{part}_rate"].median()),
    "clips_under_50pct": int((ok[f"computed_{part}_rate"] < 0.5).sum()),
    "clips_at_zero": int((ok[f"computed_{part}_rate"] == 0).sum()),
} for part in ("face", "pose", "left_hand", "right_hand")}
print("\nAGGREGATE DETECTION RATES")
for part, stats in aggregate_rates.items():
    print(f"  {part:<12} mean {stats['mean']:.3f} | median {stats['median']:.3f} | "
          f"under 50%: {stats['clips_under_50pct']:,} | always absent: {stats['clips_at_zero']:,}")

alignment = pd.DataFrame([
    {"check": "clips in both metadata files", "value": len(clips),
     "expected": EXPECTED_CLIPS, "passed": len(clips) == EXPECTED_CLIPS},
    {"check": "clips only in full", "value": len(only_full), "expected": 0,
     "passed": len(only_full) == 0},
    {"check": "clips only in reduced", "value": len(only_reduced), "expected": 0,
     "passed": len(only_reduced) == 0},
    {"check": ".npy on disk (full)", "value": len(full_files),
     "expected": EXPECTED_CLIPS, "passed": len(full_files) == EXPECTED_CLIPS},
    {"check": ".npy on disk (reduced)", "value": len(reduced_files),
     "expected": EXPECTED_CLIPS, "passed": len(reduced_files) == EXPECTED_CLIPS},
    {"check": "orphan files", "value": len(orphans_full) + len(orphans_reduced),
     "expected": 0, "passed": not (orphans_full or orphans_reduced)},
    {"check": "absent files", "value": len(absent_full) + len(absent_reduced),
     "expected": 0, "passed": not (absent_full or absent_reduced)},
    {"check": "disk path sets identical", "value": len(disk_only_full) + len(disk_only_reduced),
     "expected": 0, "passed": full_disk_set == reduced_disk_set},
    {"check": "duplicate clip ids", "value": len(duplicates_full) + len(duplicates_reduced),
     "expected": 0, "passed": not (duplicates_full or duplicates_reduced)},
    {"check": "duplicate relative paths",
     "value": len(duplicate_paths_full) + len(duplicate_paths_reduced),
     "expected": 0, "passed": not (duplicate_paths_full or duplicate_paths_reduced)},
    {"check": "frame counts match", "value": int((~ok["frames_match"]).sum()),
     "expected": 0, "passed": bool(ok["frames_match"].all())},
    {"check": "presence masks identical", "value": int((~ok["masks_identical"]).sum()),
     "expected": 0, "passed": bool(ok["masks_identical"].all())},
    {"check": "tail numerically identical",
     "value": int((~ok["tail_numerically_identical"]).sum()), "expected": 0,
     "passed": bool(ok["tail_numerically_identical"].all())},
    {"check": "tail byte identical", "value": int((~ok["tail_byte_identical"]).sum()),
     "expected": 0, "passed": bool(ok["tail_byte_identical"].all())},
    {"check": "max tail difference", "value": float(ok["tail_max_abs_diff"].max()),
     "expected": 0.0, "passed": float(ok["tail_max_abs_diff"].max()) == 0.0},
    {"check": "shared-tail statistics aligned", "value": len(tail_statistics_detail),
     "expected": 0, "passed": tail_statistics_aligned},
    {"check": "shared metadata columns agree", "value": total_disagreements,
     "expected": 0, "passed": total_disagreements == 0},
])
atomic_write(ALIGNMENT_CSV, lambda handle: alignment.to_csv(handle, index=False))
print("\nBRANCH ALIGNMENT")
print(alignment.to_string(index=False))

In [ ]:
# ============================================================
# CELL 8 — Audit plots
# ============================================================
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight", "font.size": 9})
plot_paths: List[str] = []


def save(figure, name: str) -> None:
    path = PLOT_DIR / name
    figure.savefig(path)
    plt.close(figure)
    plot_paths.append(str(path))
    print(f"  {path.name}")


figure, axis = plt.subplots(figsize=(9, 4))
axis.hist(lengths, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
axis.axvline(LONG_CLIP_THRESHOLD, color="#C44E52", linestyle="--",
             label=f"loader cap {LONG_CLIP_THRESHOLD} "
                   f"({length_stats[f'over_{LONG_CLIP_THRESHOLD}_percent']:.1f}% above)")
axis.axvline(float(np.median(lengths)), color="#55A868", linestyle=":",
             label=f"median {np.median(lengths):.0f}")
axis.set_xlabel("frames"); axis.set_ylabel("clips")
axis.set_title("Sequence length distribution"); axis.legend()
save(figure, "sequence_length_distribution.png")

order = sorted(signer_summary.index, key=lambda s: int(str(s).lstrip("p")))
positions = np.arange(len(order))

figure, axis = plt.subplots(figsize=(9, 4))
axis.bar(positions - 0.2, signer_summary.loc[order, "left_hand_rate"], 0.4,
         label="left hand", color="#4C72B0")
axis.bar(positions + 0.2, signer_summary.loc[order, "right_hand_rate"], 0.4,
         label="right hand", color="#DD8452")
axis.set_xticks(positions); axis.set_xticklabels(order, rotation=45)
axis.set_ylabel("mean detection rate"); axis.set_ylim(0, 1.05)
axis.set_title("Hand detection rate by signer"); axis.legend()
save(figure, "detection_rate_by_signer.png")

figure, axis = plt.subplots(figsize=(9, 4))
axis.bar(positions, signer_summary.loc[order, "clips"], color="#55A868")
axis.set_xticks(positions); axis.set_xticklabels(order, rotation=45)
axis.set_ylabel("clips"); axis.set_title("Clips per signer")
save(figure, "clips_per_signer.png")

figure, axis = plt.subplots(figsize=(10, 4))
axis.boxplot([ok.loc[ok["signer_id"] == signer, "full_frames"].to_numpy() for signer in order],
             showfliers=False)
axis.set_xticks(np.arange(1, len(order) + 1)); axis.set_xticklabels(order, rotation=45)
axis.axhline(LONG_CLIP_THRESHOLD, color="#C44E52", linestyle="--")
axis.set_ylabel("frames"); axis.set_title("Sequence length by signer")
save(figure, "sequence_length_by_signer.png")

figure, axis = plt.subplots(figsize=(9, 4))
for part, colour in (("face", "#4C72B0"), ("pose", "#55A868"),
                     ("left_hand", "#DD8452"), ("right_hand", "#C44E52")):
    axis.hist(ok[f"computed_{part}_rate"], bins=40, histtype="step",
              linewidth=1.6, label=part, color=colour)
axis.set_xlabel("per-clip detection rate"); axis.set_ylabel("clips")
axis.set_yscale("log"); axis.set_title("Detection rate distribution by modality")
axis.legend()
save(figure, "detection_rate_distribution.png")

In [ ]:
# ============================================================
# CELL 9 — Verdict, completeness assertions, publication
# ============================================================
successful_clip_ids = sorted(ok["clip"].astype(str))
SUCCESSFUL_CLIP_ID_HASH = hashlib.sha256("|".join(successful_clip_ids).encode()).hexdigest()[:16]
statistics_clip_ids = sorted(processed_clips)

completeness = [
    {"check": f"unique .npy on disk, full == {EXPECTED_AUDIT_COUNT if SMOKE_TEST else EXPECTED_CLIPS:,}",
     "passed": SMOKE_TEST or len(full_disk_set) == EXPECTED_CLIPS,
     "detail": f"{len(full_disk_set):,}"},
    {"check": f"unique .npy on disk, reduced == {EXPECTED_AUDIT_COUNT if SMOKE_TEST else EXPECTED_CLIPS:,}",
     "passed": SMOKE_TEST or len(reduced_disk_set) == EXPECTED_CLIPS,
     "detail": f"{len(reduced_disk_set):,}"},
    {"check": f"canonical clips == {EXPECTED_CLIPS:,}",
     "passed": SMOKE_TEST or len(clips) == EXPECTED_CLIPS, "detail": f"{len(clips):,}"},
    {"check": f"successful audit rows == {EXPECTED_AUDIT_COUNT:,}",
     "passed": len(ok) == EXPECTED_AUDIT_COUNT, "detail": f"{len(ok):,}"},
    {"check": "successful audit rows are unique",
     "passed": len(successful_clip_ids) == len(set(successful_clip_ids)),
     "detail": f"{len(set(successful_clip_ids)):,} unique"},
    {"check": "zero failed rows", "passed": len(failed_reads) == 0,
     "detail": f"{len(failed_reads):,}"},
    {"check": "statistics cover exactly the successful clips",
     "passed": statistics_clip_ids == successful_clip_ids,
     "detail": f"{len(statistics_clip_ids):,} vs {len(successful_clip_ids):,}"},
]

checks: List[Dict[str, Any]] = completeness + [
    {"check": f"{EXPECTED_LABELS} labels", "passed": clips["label"].nunique() == EXPECTED_LABELS,
     "detail": f"{clips['label'].nunique()}"},
    {"check": f"{EXPECTED_SIGNERS} signers",
     "passed": clips["signer_id"].nunique() == EXPECTED_SIGNERS,
     "detail": f"{clips['signer_id'].nunique()}"},
    {"check": "every signer covers every class", "passed": len(incomplete_signers) == 0,
     "detail": f"min {int(coverage.min())}"},
    {"check": "schemas match the expected layout", "passed": SCHEMAS_VALID, "detail": "cell 2"},
    {"check": "no duplicate clip ids or paths",
     "passed": not (duplicates_full or duplicates_reduced
                    or duplicate_paths_full or duplicate_paths_reduced), "detail": ""},
    {"check": "no orphan or absent files",
     "passed": not (orphans_full or orphans_reduced or absent_full or absent_reduced),
     "detail": f"orphan {len(orphans_full)}/{len(orphans_reduced)}, "
               f"absent {len(absent_full)}/{len(absent_reduced)}"},
    {"check": "disk path sets identical", "passed": full_disk_set == reduced_disk_set,
     "detail": f"{len(disk_only_full)}/{len(disk_only_reduced)} one-sided"},
    {"check": f"full width == {FULL.width}", "passed": bool(ok["full_width_ok"].all()),
     "detail": f"{int((~ok['full_width_ok']).sum())} bad"},
    {"check": f"reduced width == {REDUCED.width}", "passed": bool(ok["reduced_width_ok"].all()),
     "detail": f"{int((~ok['reduced_width_ok']).sum())} bad"},
    {"check": "no NaN or Inf (full)", "passed": bool(ok["full_finite"].all()),
     "detail": f"{int((~ok['full_finite']).sum())} bad"},
    {"check": "no NaN or Inf (reduced)", "passed": bool(ok["reduced_finite"].all()),
     "detail": f"{int((~ok['reduced_finite']).sum())} bad"},
    {"check": "presence masks strictly binary",
     "passed": bool(ok["full_mask_binary"].all() and ok["reduced_mask_binary"].all()),
     "detail": f"{int((~ok['full_mask_binary']).sum())}/{int((~ok['reduced_mask_binary']).sum())}"},
    {"check": "frame counts agree", "passed": bool(ok["frames_match"].all()),
     "detail": f"{int((~ok['frames_match']).sum())} mismatched"},
    {"check": "tail numerically identical",
     "passed": bool(ok["tail_numerically_identical"].all()),
     "detail": f"max diff {ok['tail_max_abs_diff'].max():.2e}"},
    {"check": "tail byte identical", "passed": bool(ok["tail_byte_identical"].all()),
     "detail": f"{int((~ok['tail_byte_identical']).sum())} differing"},
    {"check": "shared-tail statistics aligned", "passed": tail_statistics_aligned,
     "detail": f"{len(tail_statistics_detail)} signers differing"},
    {"check": f"all {len(shared_columns)} shared metadata columns agree",
     "passed": total_disagreements == 0,
     "detail": ", ".join(f"{k}:{len(v)}" for k, v in metadata_disagreements.items() if len(v))
               or "none"},
    {"check": "recomputed detection rates match metadata",
     "passed": all(s["max_abs_difference"] < 1e-6 for s in rate_agreement.values()),
     "detail": ", ".join(f"{k}:{v['max_abs_difference']:.1e}" for k, v in rate_agreement.items())},
    {"check": "per-signer statistics complete and finite", "passed": statistics_complete,
     "detail": ", ".join(f"{b}:{r['total_frames']:,}f" for b, r in statistics_report.items())},
    {"check": f"pipeline id == {EXPECTED_PIPELINE_ID}", "passed": PIPELINE_ID == EXPECTED_PIPELINE_ID,
     "detail": PIPELINE_ID},
]

print("=" * 78)
print(f"POST-NORMALISATION AUDIT  ({AUDIT_VERSION})")
print("=" * 78)
for entry in checks:
    print(f"  {'PASS' if entry['passed'] else 'FAIL'}  {entry['check']:<52} {entry['detail']}")

all_passed = all(entry["passed"] for entry in checks)
VERDICT = "PASS" if all_passed else "FAIL"

# ---- Publish statistics only after a full-mode PASS ----------------------
published: List[str] = []
if all_passed and not SMOKE_TEST:
    for branch in ("full", "reduced"):
        payload = dict(STATS_PAYLOADS[branch])
        payload["__audit_verdict__"] = np.array(["PASS"])
        payload["__successful_clip_count__"] = np.array([len(successful_clip_ids)], dtype=np.int64)
        payload["__successful_clip_id_hash__"] = np.array([SUCCESSFUL_CLIP_ID_HASH])
        atomic_write(PUBLISHED_STATS[branch],
                     lambda handle, p=payload: np.savez_compressed(handle, **p))
        published.append(PUBLISHED_STATS[branch].name)
        UNVERIFIED_STATS[branch].unlink(missing_ok=True)
else:
    # Never leave trusted-looking files behind after a FAIL or a smoke test.
    for branch in ("full", "reduced"):
        if PUBLISHED_STATS[branch].exists():
            PUBLISHED_STATS[branch].unlink()
            print(f"\n  removed stale {PUBLISHED_STATS[branch].name} "
                  "(not republished this run)")

summary = {
    "notebook": "6-post-normalization-audit-v3", "audit_version": AUDIT_VERSION,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "verdict": VERDICT, "mode": "smoke_test" if SMOKE_TEST else "full",
    "pipeline_id": PIPELINE_ID, "fold_manifest_hash": None,
    "clip_table_hash": CLIP_TABLE_HASH, "clip_table_hash_columns": HASHED_COLUMNS,
    "successful_clip_id_hash": SUCCESSFUL_CLIP_ID_HASH,
    "metadata_hashes": METADATA_HASHES, "schema_hashes": SCHEMA_HASHES,
    "block_layout_hashes": BLOCK_LAYOUT_HASHES,
    "environment": ENVIRONMENT,
    "paths": {"full_root": str(FULL_BUNDLE["root"]), "reduced_root": str(REDUCED_BUNDLE["root"]),
              "full_dir": str(FULL_DIR), "reduced_dir": str(REDUCED_DIR)},
    "layout": {"full": asdict(FULL) | {"root": str(FULL.root)},
               "reduced": asdict(REDUCED) | {"root": str(REDUCED.root)}},
    "counts": {"clips_in_both": len(clips), "clips_audited": len(ok),
               "expected_audit_count": EXPECTED_AUDIT_COUNT,
               "labels": int(clips["label"].nunique()),
               "signers": int(clips["signer_id"].nunique()),
               "read_failures": len(failed_reads),
               "files_on_disk_full": len(full_files), "files_on_disk_reduced": len(reduced_files),
               "orphans_full": orphans_full[:50], "orphans_reduced": orphans_reduced[:50],
               "absent_full": absent_full[:50], "absent_reduced": absent_reduced[:50],
               "only_full": only_full, "only_reduced": only_reduced},
    "sequence_lengths": length_stats, "detection_rates_aggregate": aggregate_rates,
    "detection_rate_agreement": rate_agreement,
    "branch_alignment": alignment.to_dict("records"),
    "signer_feature_statistics": statistics_report,
    "shared_tail_statistics_aligned": tail_statistics_aligned,
    "shared_metadata_columns_compared": shared_columns,
    "completeness": completeness, "checks": checks,
    "published_statistics": published,
    "outputs": {"audit_csv": str(AUDIT_CSV), "audit_json": str(AUDIT_JSON),
                "signer_summary_csv": str(SIGNER_CSV), "alignment_csv": str(ALIGNMENT_CSV),
                "plots": plot_paths},
}
atomic_write(AUDIT_JSON,
             lambda handle: handle.write(json.dumps(summary, indent=2, default=str).encode()))

working_bytes = sum(p.stat().st_size for p in WORKING.rglob("*") if p.is_file())
print("\n" + "=" * 78)
if all_passed and SMOKE_TEST:
    print("VERDICT: PASS (SMOKE TEST ONLY)")
    print("=" * 78)
    print("  *** THIS IS NOT PERMISSION TO START NOTEBOOK 7. ***")
    print(f"  Only {len(ok):,} clips were audited and statistics were NOT published.")
    print("  Set SMOKE_TEST = False and rerun a full pass first.")
elif all_passed:
    print("VERDICT: PASS - both branches are complete, valid and aligned.")
    print("=" * 78)
    print(f"  published: {', '.join(published)}")
else:
    print("VERDICT: FAIL - resolve the failing checks above before notebook 7.")
    print("=" * 78)
    print("  Statistics were NOT published.")

print(f"\n  clip table hash        : {CLIP_TABLE_HASH}")
print(f"  successful clip hash   : {SUCCESSFUL_CLIP_ID_HASH}")
print(f"  pipeline id            : {PIPELINE_ID}")
print("\n  Files written:")
for path in (AUDIT_CSV, AUDIT_JSON, SIGNER_CSV, ALIGNMENT_CSV):
    print(f"    {path.name}")
for name in published:
    print(f"    {name}")
for path in plot_paths:
    print(f"    audit_plots/{Path(path).name}")
print(f"\n  /kaggle/working total: {working_bytes / (1024 ** 3):.2f} GB of the 20 GB cap")